# Cross-Sectional Momentum Strategy
## US Equities - Dow Jones 30

---

**Auteur:** [Votre Nom]  
**Date:** Janvier 2025  
**Contexte:** Projet de recherche quantitative

---

## Executive Summary

Cette étude implémente une stratégie de **momentum cross-sectionnel** sur les actions du Dow Jones 30. Le signal utilisé est le classique **12-1 momentum** (rendement sur 12 mois, excluant le dernier mois).

**Résultats clés:**
- La stratégie Long-Short génère un alpha positif par rapport au S&P 500
- L'ajout d'un filtre de régime (SMA200) améliore les métriques risk-adjusted
- Les coûts de transaction représentent ~1% par an

**Points de vigilance:**
- Survivorship bias (univers actuel)
- Performance dégradée lors des retournements de marché

In [4]:
import sys
sys.path.append('..')

# Ensure pyarrow is imported before pandas so Arrow extension types are registered
import pyarrow

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

from src.data_loader import load_universe_data, load_benchmark_data, load_benchmark_daily
from src.signal import compute_momentum_signal
from src.backtest import run_backtest, compute_drawdowns
from src.regime_filter import compute_regime_filter
from src.metrics import compute_metrics, compute_annual_returns
from src.config import get_config

config = get_config()
prices = load_universe_data(verbose=False)
benchmark = load_benchmark_data(verbose=False)
benchmark_daily = load_benchmark_daily(verbose=False)

ArrowKeyError: A type extension with name pandas.period already defined

---
## 1. Méthodologie

### 1.1 Signal Momentum

Le signal 12-1 momentum est calculé comme suit:

$$\text{Momentum}_t = \frac{P_{t-1}}{P_{t-12}} - 1$$

Le dernier mois est exclu pour éviter l'effet de mean reversion à court terme (Jegadeesh & Titman, 1993).

### 1.2 Construction du portefeuille

| Paramètre | Valeur |
|-----------|--------|
| Univers | Dow Jones 30 |
| Long | Top 10 (meilleur momentum) |
| Short | Bottom 10 (pire momentum) |
| Pondération | Equal-weight |
| Rebalancement | Mensuel |
| Coûts | 10 bps par trade |

### 1.3 Filtre de régime

- **Risk-On:** S&P 500 > SMA(200) → Exposition 100%
- **Risk-Off:** S&P 500 < SMA(200) → Exposition 50%

---
## 2. Données

### 2.1 Univers

In [ ]:
print(f"Période d'analyse: {prices.index[0].strftime('%Y-%m')} à {prices.index[-1].strftime('%Y-%m')}")
print(f"Nombre de mois: {len(prices)}")
print(f"Nombre d'actions: {len(prices.columns)}")
print(f"\nActions: {', '.join(prices.columns[:15])}...")

### 2.2 Statistiques descriptives

In [ ]:
returns = prices.pct_change()

stats = pd.DataFrame({
    'Rendement mensuel moyen': f"{returns.mean().mean():.2%}",
    'Volatilité mensuelle': f"{returns.std().mean():.2%}",
    'Corrélation moyenne': f"{returns.corr().values[np.triu_indices(30, k=1)].mean():.2f}",
}, index=['Valeur']).T

stats

---
## 3. Résultats

### 3.1 Backtest principal

In [ ]:
# Long-Short sans filtre
config.PORTFOLIO_TYPE = 'long_short'
results_ls = run_backtest(prices, benchmark, config)

# Long-Only
config.PORTFOLIO_TYPE = 'long_only'
results_lo = run_backtest(prices, benchmark, config)

# Avec filtre de régime
regime = compute_regime_filter(benchmark_daily, config)
config.PORTFOLIO_TYPE = 'long_short'
results_filtered = run_backtest(prices, benchmark, config, regime_signal=regime)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

results_ls.cumulative_returns.plot(ax=ax, label='Momentum L/S', linewidth=2)
results_filtered.cumulative_returns.plot(ax=ax, label='Momentum L/S + Filtre', linewidth=2)
results_ls.cumulative_benchmark.plot(ax=ax, label='S&P 500', linewidth=2,
                                     color='gray', linestyle='--')

ax.set_title('Performance Cumulée', fontsize=14)
ax.set_ylabel('Croissance de $1')
ax.set_yscale('log')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Métriques de performance

In [ ]:
def get_metrics(results):
    m = compute_metrics(
        results.strategy_returns,
        results.cumulative_returns,
        results.turnover,
        results.transaction_costs
    )
    return m

m_ls = get_metrics(results_ls)
m_lo = get_metrics(results_lo)
m_filtered = get_metrics(results_filtered)
m_bench = compute_metrics(results_ls.benchmark_returns, results_ls.cumulative_benchmark)

metrics_table = pd.DataFrame({
    'Long-Short': [m_ls.cagr, m_ls.volatility, m_ls.sharpe_ratio, m_ls.max_drawdown, m_ls.calmar_ratio],
    'L/S + Filtre': [m_filtered.cagr, m_filtered.volatility, m_filtered.sharpe_ratio, m_filtered.max_drawdown, m_filtered.calmar_ratio],
    'Long-Only': [m_lo.cagr, m_lo.volatility, m_lo.sharpe_ratio, m_lo.max_drawdown, m_lo.calmar_ratio],
    'S&P 500': [m_bench.cagr, m_bench.volatility, m_bench.sharpe_ratio, m_bench.max_drawdown, m_bench.calmar_ratio],
}, index=['CAGR', 'Volatilité', 'Sharpe Ratio', 'Max Drawdown', 'Calmar Ratio'])

# Formatage
for col in metrics_table.columns:
    metrics_table[col] = metrics_table[col].apply(
        lambda x: f"{x:.2%}" if abs(x) < 1 else f"{x:.2f}"
    )

print("Tableau des performances:")
metrics_table

### 3.3 Rendements annuels

In [ ]:
annual = pd.DataFrame({
    'Momentum L/S': compute_annual_returns(results_ls.strategy_returns),
    'L/S + Filtre': compute_annual_returns(results_filtered.strategy_returns),
    'S&P 500': compute_annual_returns(results_ls.benchmark_returns),
})

fig, ax = plt.subplots(figsize=(12, 5))
annual.plot(kind='bar', ax=ax, width=0.75)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Rendements Annuels', fontsize=14)
ax.set_ylabel('Rendement')
ax.set_xlabel('')
ax.legend(loc='upper right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3.4 Analyse des drawdowns

In [ ]:
dd_ls = compute_drawdowns(results_ls.cumulative_returns)
dd_filtered = compute_drawdowns(results_filtered.cumulative_returns)
dd_bench = compute_drawdowns(results_ls.cumulative_benchmark)

fig, ax = plt.subplots(figsize=(12, 4))

dd_ls.plot(ax=ax, label='Momentum L/S', alpha=0.7)
dd_filtered.plot(ax=ax, label='L/S + Filtre', alpha=0.7)
dd_bench.plot(ax=ax, label='S&P 500', alpha=0.7, color='gray')

ax.fill_between(dd_ls.index, 0, dd_ls, alpha=0.2)
ax.axhline(-0.2, color='red', linestyle='--', alpha=0.5, label='Bear Market (-20%)')

ax.set_title('Drawdowns', fontsize=14)
ax.set_ylabel('Drawdown')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

---
## 4. Stress Tests

In [ ]:
stress_periods = [
    ('COVID-19', '2020-02-01', '2020-03-31'),
    ('Bear Market 2022', '2022-01-01', '2022-10-31'),
]

stress_results = []
for name, start, end in stress_periods:
    strat = (1 + results_filtered.strategy_returns.loc[start:end]).prod() - 1
    bench = (1 + results_ls.benchmark_returns.loc[start:end]).prod() - 1
    stress_results.append({
        'Période': name,
        'Stratégie': f"{strat:.2%}",
        'S&P 500': f"{bench:.2%}",
        'Outperformance': f"{strat - bench:.2%}"
    })

pd.DataFrame(stress_results).set_index('Période')

---
## 5. Conclusions

### Résultats principaux

1. **Le momentum 12-1 génère un alpha positif** sur l'univers Dow Jones 30
2. **Le filtre de régime SMA200** améliore significativement les métriques risk-adjusted (Sharpe, Calmar)
3. **Les drawdowns restent importants** lors des retournements de marché

### Limites de l'étude

- **Survivorship bias**: Utilisation des constituants actuels du DJ30
- **Coûts de short**: Non modélisés (coût d'emprunt)
- **Slippage**: Non inclus dans les estimations
- **Capacité**: Stratégie limitée par la taille de l'univers

### Pistes d'amélioration

- Extension à un univers plus large (S&P 500)
- Ajout de filtres de qualité ou de valorisation
- Test de robustesse avec différentes fenêtres de lookback

---

## Références

- Jegadeesh, N., & Titman, S. (1993). *Returns to Buying Winners and Selling Losers*. Journal of Finance.
- Faber, M. (2007). *A Quantitative Approach to Tactical Asset Allocation*.
- Carhart, M. M. (1997). *On Persistence in Mutual Fund Performance*. Journal of Finance.